# 01 - Dataset Exploration

Counts the images in every class folder of `ml/dataset/raw/` (filled by `ml/dataset/download_dataset.py`) and shows a sorted summary table. Empty class folders are flagged in red.

Works in **Google Colab** and locally (VS Code). In Colab it reuses the Drive-mount / path-setup cells of `00_colab_setup.ipynb`, so `PROJECT_PATH` is only ever edited there.

In [ ]:
# Cell 1: reuse the Drive-mount / path-setup cells of 00_colab_setup.ipynb (Prompt 4) - that logic is NOT duplicated here.
# Result: PROJECT_PATH is defined, Drive is mounted (Colab) and the working directory is the project folder.
import glob
import json
import os
from pathlib import Path

SETUP_NOTEBOOK = None  # None = auto-find 00_colab_setup.ipynb on Drive; set a full path only if auto-find fails

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    DRIVE_ROOT = "/content/drive/MyDrive"
    if not os.path.isdir(DRIVE_ROOT):
        # Bootstrap only: Drive must be reachable before the shared setup notebook can be found.
        from google.colab import drive
        drive.mount("/content/drive")

    if SETUP_NOTEBOOK is None:
        found = []
        for depth in range(1, 4):  # look up to 3 folders deep inside My Drive
            found += glob.glob(os.path.join(DRIVE_ROOT, *["*"] * depth, "ml", "notebooks", "00_colab_setup.ipynb"))
        if len(found) != 1:
            raise FileNotFoundError(
                f"Expected exactly one 00_colab_setup.ipynb on your Drive, found {len(found)}: {found}\n"
                "Fix: upload the project to Drive, or set SETUP_NOTEBOOK above to the notebook's full path."
            )
        SETUP_NOTEBOOK = found[0]

    # Run the first two code cells (Drive mount + path setup) of the setup notebook.
    with open(SETUP_NOTEBOOK) as f:
        setup_cells = [c for c in json.load(f)["cells"] if c["cell_type"] == "code"][:2]
    for cell in setup_cells:
        exec(compile("".join(cell["source"]), SETUP_NOTEBOOK, "exec"), globals())
else:
    # Local run (e.g. VS Code on a Mac): the setup notebook is Colab-only, so just find the project root.
    start = Path.cwd().resolve()
    root = next((p for p in [start, *start.parents] if (p / "ml" / "dataset").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Open this notebook from inside your crop-disease-detection project (a folder containing ml/dataset/).")
    PROJECT_PATH = str(root)
    os.chdir(PROJECT_PATH)
    print("Working directory:", os.getcwd())

In [ ]:
# Cell 2: count images per class folder and show a sorted, colour-coded table
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

RAW_DIR = Path(PROJECT_PATH) / "ml" / "dataset" / "raw"   # filled by ml/dataset/download_dataset.py (Prompt 6)
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}                    # matched case-insensitively

if not RAW_DIR.is_dir():
    raise FileNotFoundError(
        f"{RAW_DIR} not found.\n"
        "Fix: run the Prompt 6 script first:  python ml/dataset/download_dataset.py"
    )

with os.scandir(RAW_DIR) as top:
    entries = sorted((e.name, e.path, e.is_dir()) for e in top)

rows, subdir_counts = [], {}
non_image_files = loose_files = 0
for name, path, is_dir in entries:
    if name.startswith("."):
        continue
    if not is_dir:
        loose_files += 1                      # a file sitting directly in raw/, not in a class folder
        continue
    images = others = subdirs = 0
    with os.scandir(path) as it:
        for f in it:
            if f.is_file():
                if os.path.splitext(f.name)[1].lower() in IMAGE_EXTS:
                    images += 1
                else:
                    others += 1
            elif f.is_dir():
                subdirs += 1
    non_image_files += others
    subdir_counts[name] = subdirs
    rows.append({"class_name": name, "image_count": images})

if not rows:
    raise RuntimeError(f"{RAW_DIR} has no class sub-folders. Run: python ml/dataset/download_dataset.py --force")

df = (pd.DataFrame(rows, columns=["class_name", "image_count"])
        .sort_values(["image_count", "class_name"], ascending=[False, True])
        .reset_index(drop=True))


def flag(row):
    if row["image_count"] > 0:
        return ""
    if subdir_counts[row["class_name"]]:
        return "EMPTY - only sub-folders (unexpected layout)"
    return "EMPTY - 0 images (failed or partial download?)"


table = df.assign(flag=df.apply(flag, axis=1))


def highlight_empty(row):
    return ["background-color: #f4a6a6; font-weight: bold" if row["image_count"] == 0 else "" for _ in row]


display(
    table.style
    .background_gradient(subset=["image_count"], cmap="YlGn")
    .apply(highlight_empty, axis=1)
    .format({"image_count": "{:,}"})
    .hide(axis="index")
)

In [ ]:
# Cell 3: summary
total_images = int(df["image_count"].sum())
n_classes = len(df)
empty = df[df["image_count"] == 0]
non_empty = df[df["image_count"] > 0]

if len(empty):
    print(f"WARNING: {len(empty)} class folder(s) contain ZERO images: {', '.join(empty['class_name'])}")
    print("         Re-run: python ml/dataset/download_dataset.py --force")
else:
    print("OK: every class folder contains at least one image.")

if len(non_empty):
    big, small = non_empty.iloc[0], non_empty.iloc[-1]
    print(f"Largest class : {big['class_name']} ({big['image_count']:,})")
    print(f"Smallest class: {small['class_name']} ({small['image_count']:,})  ->  imbalance ratio {big['image_count'] / small['image_count']:.1f}x")

if non_image_files or loose_files:
    print(f"Note: {non_image_files:,} non-image file(s) inside class folders and {loose_files:,} loose file(s) in raw/ were not counted as images.")

# Reconciliation with the shell: images + non-image + loose files  ==  find ml/dataset/raw -type f | wc -l
# (files nested deeper than one level inside a class folder are not counted)
print(f"Total files seen: {total_images + non_image_files + loose_files:,}  (compare with: find ml/dataset/raw -type f | wc -l)")

print(f"\nSUMMARY: {total_images:,} images across {n_classes} classes")